# Session 4: LangSmith - Observability & Debugging (45 minutes)

## 🎯 Learning Objectives
- Understand why observability is critical in production
- Set up LangSmith for tracing
- Debug LLM applications effectively
- Monitor and evaluate chains

## 📋 Problem Statement
Our RAG system is working, but:
- How do we debug when things go wrong?
- How do we measure quality?
- How do we optimize costs?

## ⏱️ Session Breakdown
- 5 min: Why observability matters
- 10 min: LangSmith setup
- 15 min: Tracing and debugging
- 10 min: Evaluation concepts
- 5 min: Recap

---

## 🆓 Using qwen2:0.5b

## 1. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Initialize TinyLlama via Ollama (LOCAL, FREE!)
llm = ChatOllama(
    model="qwen2:0.5b",
    temperature=0.7
)

print("✅ Session 4 Setup Complete!")
print("🖥️  Using TinyLlama via Ollama - LOCAL, FREE, NO LIMITS!")

✅ Session 4 Setup Complete!
🖥️  Using TinyLlama via Ollama - LOCAL, FREE, NO LIMITS!


## 2. Why Observability Matters

🎯 **The LLM Debugging Challenge:**

Unlike traditional code:
- **Non-deterministic**: Same input → different outputs
- **Complex pipelines**: Multiple LLM calls, retrievers, tools
- **Hidden costs**: Token usage adds up quickly
- **Subtle failures**: Wrong answers that look correct

```
Traditional Debugging:     LLM Debugging:
├── Set breakpoint         ├── What prompt was sent?
├── Check variable         ├── What did the LLM return?
├── Step through code      ├── Which docs were retrieved?
└── Find bug               ├── How many tokens used?
                           └── Why is the answer wrong?
```

## 3. Setting Up LangSmith

LangSmith provides:
- **Tracing**: See every step of your chain
- **Debugging**: Understand failures
- **Evaluation**: Measure quality
- **Monitoring**: Track production performance

In [ ]:
# Enable LangSmith tracing (set these in your .env file)
# LANGCHAIN_TRACING_V2=true
# LANGCHAIN_API_KEY=your-langsmith-api-key
# LANGCHAIN_PROJECT=genai-workshop

# Check if LangSmith is configured
tracing_enabled = os.getenv("LANGCHAIN_TRACING_V2", "false").lower() == "true"
langsmith_key = os.getenv("LANGCHAIN_API_KEY", "")
project = os.getenv("LANGCHAIN_PROJECT", "default")

print("🔍 LangSmith Configuration:")
print(f"  Tracing Enabled: {tracing_enabled}")
print(f"  API Key Set: {'✅' if langsmith_key else '❌'}")
print(f"  Project: {project}")

if not tracing_enabled:
    print("\n⚠️ LangSmith tracing not enabled. To enable:")
    print("  1. Go to https://smith.langchain.com")
    print("  2. Create an account (free tier available)")
    print("  3. Create an API key")
    print("  4. Add to your .env file:")
    print("     LANGCHAIN_TRACING_V2=true")
    print("     LANGCHAIN_API_KEY=your-key")
    print("     LANGCHAIN_PROJECT=genai-workshop")

## 4. Enable Tracing Programmatically

In [2]:
# You can also enable tracing programmatically
def enable_langsmith_tracing(project_name: str = "genai-workshop"):
    """Enable LangSmith tracing."""
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_PROJECT"] = project_name
    print(f"✅ LangSmith tracing enabled for project: {project_name}")

def disable_langsmith_tracing():
    """Disable LangSmith tracing."""
    os.environ["LANGCHAIN_TRACING_V2"] = "false"
    print("❌ LangSmith tracing disabled")

# Enable tracing for this session (uncomment if API key is set)
# enable_langsmith_tracing()

## 5. Understanding Traces

A **trace** captures the entire execution of a chain:

```
Trace: research_assistant_query
├── Input: "What is quantum computing?"
├── Chain: RAGChain
│   ├── Retriever: VectorStoreRetriever
│   │   ├── Input: "What is quantum computing?"
│   │   ├── Output: [Doc1, Doc2, Doc3]
│   │   └── Duration: 45ms
│   ├── Prompt: ChatPromptTemplate
│   │   └── Formatted prompt with context
│   └── LLM: ChatGoogleGenerativeAI
│       ├── Input: Formatted prompt
│       ├── Output: "Quantum computing is..."
│       ├── Tokens: Input=1200, Output=150
│       └── Duration: 2.3s
├── Output: "Quantum computing is..."
└── Total Duration: 2.5s
```

## 6. Creating Traceable Chains

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# Create a simple chain that will be traced
research_prompt = ChatPromptTemplate.from_template("""
You are a research assistant. Explain the following topic:
Topic: {topic}

Provide a clear, concise explanation suitable for a beginner.
""")

research_chain = research_prompt | llm | StrOutputParser()

# Run the chain - this will be traced if LangSmith is enabled
result = research_chain.invoke({"topic": "neural networks"})
print("🔬 Research Result:")
print(result)

print("\n📊 If LangSmith is enabled, check your dashboard for the trace!")

## 7. Adding Custom Metadata to Traces

In [ ]:
from langchain_core.runnables import RunnableConfig

# Add metadata and tags to traces
config = RunnableConfig(
    tags=["workshop", "session-4", "demo"],
    metadata={
        "user_id": "participant-001",
        "session": "langsmith-intro",
        "version": "1.0"
    },
    run_name="research_assistant_query"  # Custom name in LangSmith
)

# Run with config
result = research_chain.invoke(
    {"topic": "transformer architecture"},
    config=config
)

print("🏷️ Trace created with custom tags and metadata!")
print(result[:200] + "...")

## 8. Manual Tracing with Callbacks

In [ ]:
from langchain_core.callbacks import StdOutCallbackHandler
from langchain_core.tracers import ConsoleCallbackHandler

# Console tracing (useful for local debugging)
console_callback = ConsoleCallbackHandler()

print("🖥️ Running with console tracing...\n")

# This will print trace info to console
simple_prompt = ChatPromptTemplate.from_template("Tell me a short joke about {topic}")
simple_chain = simple_prompt | llm | StrOutputParser()

result = simple_chain.invoke(
    {"topic": "programming"},
    config={"callbacks": [console_callback]}
)

print(f"\n😄 Result: {result}")

## 9. Debugging Multi-Step Chains

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel

# Create a complex chain to demonstrate tracing
def log_step(step_name):
    """Create a logging function for debugging."""
    def _log(x):
        print(f"📍 {step_name}: {str(x)[:100]}...")
        return x
    return RunnableLambda(_log)

# Multi-step chain with logging
analysis_chain = (
    log_step("Input")
    | ChatPromptTemplate.from_template("Summarize this topic in 2 sentences: {topic}")
    | log_step("After Prompt")
    | llm
    | log_step("After LLM")
    | StrOutputParser()
    | log_step("Final Output")
)

print("🔍 Tracing multi-step chain execution:\n")
result = analysis_chain.invoke({"topic": "artificial intelligence"})

## 10. Understanding LangSmith Dashboard

When you have traces in LangSmith, you can:

### View Trace Details
- See exact prompts sent to LLM
- View token counts and costs
- Check latency per step

### Filter and Search
- By tags: `tags:production`
- By metadata: `metadata.user_id:123`
- By status: `status:error`

### Analyze Performance
- Average latency over time
- Token usage patterns
- Error rates

In [ ]:
# Example: Tag traces for different environments
def get_trace_config(environment: str, user_id: str = None):
    """Generate trace configuration for different environments."""
    tags = [environment]
    metadata = {"environment": environment}
    
    if user_id:
        metadata["user_id"] = user_id
        tags.append(f"user:{user_id}")
    
    return RunnableConfig(
        tags=tags,
        metadata=metadata,
        run_name=f"{environment}_query"
    )

# Different configs for different scenarios
dev_config = get_trace_config("development", "dev-user")
prod_config = get_trace_config("production", "user-12345")
test_config = get_trace_config("testing")

print("📊 Trace configurations created:")
print(f"  Dev: {dev_config.tags}")
print(f"  Prod: {prod_config.tags}")
print(f"  Test: {test_config.tags}")

## 11. Evaluation Concepts

LangSmith enables systematic evaluation of LLM outputs:

### Dataset-Based Evaluation
1. Create a dataset of (input, expected_output) pairs
2. Run your chain on each input
3. Compare outputs using evaluators

### Common Evaluators
- **Correctness**: Is the answer right?
- **Relevance**: Is the answer relevant to the question?
- **Faithfulness**: Is the answer grounded in the context?
- **Helpfulness**: Is the answer useful?

In [ ]:
# Example: Simple evaluation function
def evaluate_response(
    question: str,
    response: str,
    expected_answer: str = None,
    context: str = None
):
    """Evaluate a response using LLM-as-judge."""
    
    eval_prompt = ChatPromptTemplate.from_template("""
    Evaluate the following response for quality.
    
    Question: {question}
    Response: {response}
    {expected_section}
    {context_section}
    
    Rate the response on these criteria (1-5):
    1. Relevance: Does it answer the question?
    2. Accuracy: Is the information correct?
    3. Clarity: Is it easy to understand?
    4. Completeness: Does it fully address the question?
    
    Provide ratings and brief justification for each.
    """)
    
    expected_section = f"Expected Answer: {expected_answer}" if expected_answer else ""
    context_section = f"Context: {context}" if context else ""
    
    eval_chain = eval_prompt | llm | StrOutputParser()
    
    evaluation = eval_chain.invoke({
        "question": question,
        "response": response,
        "expected_section": expected_section,
        "context_section": context_section
    })
    
    return evaluation

## 12. Running an Evaluation

In [ ]:
# Test the evaluation function
test_question = "What is machine learning?"
test_response = research_chain.invoke({"topic": "machine learning"})

print("🔬 Running Evaluation...\n")
print(f"Question: {test_question}")
print(f"\nResponse: {test_response[:200]}...")

evaluation = evaluate_response(test_question, test_response)
print(f"\n📊 Evaluation Results:\n{evaluation}")

## 13. Creating Test Datasets

In [ ]:
# Example test dataset
test_dataset = [
    {
        "input": {"topic": "neural networks"},
        "expected_keywords": ["neurons", "layers", "training", "learning"]
    },
    {
        "input": {"topic": "natural language processing"},
        "expected_keywords": ["text", "language", "understanding", "NLP"]
    },
    {
        "input": {"topic": "computer vision"},
        "expected_keywords": ["image", "visual", "recognition", "detection"]
    }
]

def run_test_suite(chain, test_cases):
    """Run a chain against test cases."""
    results = []
    
    for i, test_case in enumerate(test_cases, 1):
        print(f"\n🧪 Test Case {i}: {test_case['input']['topic']}")
        
        # Run the chain
        response = chain.invoke(test_case["input"])
        
        # Check for expected keywords
        response_lower = response.lower()
        keywords_found = [
            kw for kw in test_case["expected_keywords"]
            if kw.lower() in response_lower
        ]
        
        score = len(keywords_found) / len(test_case["expected_keywords"])
        
        results.append({
            "topic": test_case["input"]["topic"],
            "score": score,
            "keywords_found": keywords_found
        })
        
        print(f"  Score: {score:.0%} ({len(keywords_found)}/{len(test_case['expected_keywords'])} keywords)")
        print(f"  Found: {keywords_found}")
    
    return results

# Run the test suite
print("🧪 RUNNING TEST SUITE")
print("=" * 50)
results = run_test_suite(research_chain, test_dataset)

# Summary
avg_score = sum(r["score"] for r in results) / len(results)
print(f"\n📊 Overall Score: {avg_score:.0%}")

## 14. Debugging Common Issues

### Issue 1: Retrieval Problems

In [ ]:
# Debug retrieval by logging what was retrieved
def debug_retrieval(retriever, query):
    """Debug what documents are being retrieved."""
    print(f"🔍 Query: {query}\n")
    
    # This would be your actual retriever
    # docs = retriever.invoke(query)
    
    # Simulated for demo
    print("📄 Retrieved Documents:")
    print("  1. Document about X (score: 0.85)")
    print("  2. Document about Y (score: 0.72)")
    print("  3. Document about Z (score: 0.68)")
    print("\n⚠️ Check if these are the RIGHT documents for your query!")

debug_retrieval(None, "What is the vacation policy?")

### Issue 2: Prompt Problems

In [ ]:
# Debug the actual prompt being sent to the LLM
def debug_prompt(chain, input_data):
    """Print the formatted prompt without calling the LLM."""
    # For prompt templates, we can format without invoking
    prompt = ChatPromptTemplate.from_template("""
    You are a helpful assistant.
    
    Context: {context}
    Question: {question}
    
    Answer:""")
    
    formatted = prompt.format_messages(
        context="This is the context from retrieved docs...",
        question="What is the policy?"
    )
    
    print("📝 Formatted Prompt:")
    for msg in formatted:
        print(f"  [{msg.type}]: {msg.content}")

debug_prompt(None, {})

### Issue 3: Token Usage

In [ ]:
# Track token usage
from langchain_core.callbacks import BaseCallbackHandler

class TokenUsageCallback(BaseCallbackHandler):
    """Callback to track token usage."""
    
    def __init__(self):
        self.total_tokens = 0
        self.prompt_tokens = 0
        self.completion_tokens = 0
    
    def on_llm_end(self, response, **kwargs):
        """Track tokens when LLM call completes."""
        # Token info varies by provider
        if hasattr(response, 'llm_output') and response.llm_output:
            usage = response.llm_output.get('token_usage', {})
            self.prompt_tokens += usage.get('prompt_tokens', 0)
            self.completion_tokens += usage.get('completion_tokens', 0)
            self.total_tokens += usage.get('total_tokens', 0)
    
    def report(self):
        """Print token usage report."""
        print("📊 Token Usage Report:")
        print(f"  Prompt tokens: {self.prompt_tokens}")
        print(f"  Completion tokens: {self.completion_tokens}")
        print(f"  Total tokens: {self.total_tokens}")

# Usage example
token_tracker = TokenUsageCallback()
result = research_chain.invoke(
    {"topic": "deep learning"},
    config={"callbacks": [token_tracker]}
)

token_tracker.report()
print("\n💡 Note: Token tracking varies by LLM provider")

## 15. Best Practices Summary

In [ ]:
print("""
🎯 LANGSMITH BEST PRACTICES
════════════════════════════════════════════════════════════════

1. ALWAYS ENABLE TRACING IN PRODUCTION
   - Set LANGCHAIN_TRACING_V2=true
   - You can't debug what you can't see

2. USE MEANINGFUL TAGS & METADATA
   - Environment: dev/staging/prod
   - User ID for user-specific debugging
   - Version for A/B testing

3. NAME YOUR RUNS
   - Use run_name for easier search
   - "research_query" vs "unknown_chain"

4. CREATE TEST DATASETS
   - Build golden datasets
   - Run evaluations before deploying

5. MONITOR KEY METRICS
   - Latency per step
   - Token usage trends
   - Error rates

6. REVIEW TRACES REGULARLY
   - Spot patterns in failures
   - Identify optimization opportunities

════════════════════════════════════════════════════════════════
""")

## 📚 Session 4 Recap

### Key Takeaways:

1. **Observability is Essential**
   - LLM apps are non-deterministic and complex
   - You can't debug what you can't see

2. **LangSmith Setup:**
   ```python
   os.environ["LANGCHAIN_TRACING_V2"] = "true"
   os.environ["LANGCHAIN_API_KEY"] = "your-key"
   ```

3. **Traces Capture:**
   - Input/output at each step
   - Token usage
   - Latency
   - Custom metadata

4. **Evaluation:**
   - Build test datasets
   - Use LLM-as-judge or custom evaluators
   - Track quality over time

---

### 🔜 Next Session: LangGraph - Building Agents
"We've built chains, but what about dynamic decision-making?"

In [ ]:
print("""
╔═══════════════════════════════════════════════════════════════════════════╗
║                    SESSION 4 COMPLETE! 🎉                                  ║
║                                                                            ║
║  ☕ SHORT BREAK - 5 MINUTES ☕                                              ║
║                                                                            ║
║  Next: Session 5 - LangGraph Agents                                        ║
║  File: 05_langgraph_agents.ipynb                                           ║
║                                                                            ║
║  "Our chains are linear. But what if the LLM needs to                      ║
║   make decisions and take different paths?"                                ║
╚═══════════════════════════════════════════════════════════════════════════╝
""")